In [1]:
import os

import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from utils.cfd_wrapper import OpenFoamWrapper

#### Load Data

In [2]:
mda_df = pd.read_csv('outputs/mda_df.csv')
mda_df = mda_df.round(2)
mda_df = mda_df.iloc[:10]

#### 01 Inputs

In [3]:
cases_dir = 'outputs/openfoam_cases'
templates_dir = 'inputs/templates/openfoam'
outputs_dir = 'outputs/openfoam'

In [4]:
mda_df['tpsoft'] = 2 * mda_df['tp']
mda_df['depth'] = 15 + mda_df['swl']
mda_df['lowfreqcutoff'] = 1 / ( 3 * mda_df['tp'])
mda_df['uppfreqcutoff'] = 3/  mda_df['tp']

In [ ]:
metamodel_parameters = mda_df[['hs', 'tp', 'swl', 'tpsoft', 'depth', 'lowfreqcutoff', 'uppfreqcutoff']].to_dict(orient="list")

fixed_parameters = {'points_per_wavelenght':100,
                    'alpha_inlet_patch_vals':[],
                    'boundary_file':'/home/alonsoap_foam/OpenFOAM/alonsoap_foam-v1912/run/HyCFD/inputs/templates/openfoam/constant/polyMesh/boundary',
                    'block_mesh_dict':'/home/alonsoap_foam/OpenFOAM/alonsoap_foam-v1912/run/HyCFD/inputs/templates/openfoam/constant/polyMesh/blockMeshDict',
                    'preprocess_script':'/home/alonsoap_foam/OpenFOAM/alonsoap_foam-v1912/run/HyCFD/inputs/scripts_openfoam/preprocess_case.sh',
                    'postprocess_script':'/home/alonsoap_foam/OpenFOAM/alonsoap_foam-v1912/run/HyCFD/inputs/scripts_openfoam/postprocess_case.sh'}

openfoam_wrapper = OpenFoamWrapper(
    templates_dir = templates_dir,
    metamodel_parameters = metamodel_parameters,
    fixed_parameters = fixed_parameters,
    output_dir = cases_dir,
)

2026-02-25 05:34:32,429 - OpenFoamWrapper - WARNING - Parameter hs is not in the default_parameters
2026-02-25 05:34:32,430 - OpenFoamWrapper - WARNING - Parameter tp is not in the default_parameters
2026-02-25 05:34:32,430 - OpenFoamWrapper - WARNING - Parameter swl is not in the default_parameters
2026-02-25 05:34:32,430 - OpenFoamWrapper - WARNING - Parameter tpsoft is not in the default_parameters
2026-02-25 05:34:32,431 - OpenFoamWrapper - WARNING - Parameter depth is not in the default_parameters
2026-02-25 05:34:32,431 - OpenFoamWrapper - WARNING - Parameter lowfreqcutoff is not in the default_parameters
2026-02-25 05:34:32,431 - OpenFoamWrapper - WARNING - Parameter uppfreqcutoff is not in the default_parameters


In [6]:
openfoam_wrapper.build_cases()

FileNotFoundError: [Errno 2] No such file or directory: '/home/alonsoap_foam/OpenFOAM/alonsoap_foam-v1912/run/HyCFD/inputs/templates/openfoam/constant/polyMesh/blockMeshDict'

In [ ]:
openfoam_wrapper.save_model(model_path=os.path.join(outputs_dir,'openfoam_model.pkl'), exclude_attributes=['_env'])

In [ ]:
mda_df

,hs,hs_l0,swl,tp,tpsoft,depth,lowfreqcutoff,uppfreqcutoff
0,4.32,0.02,1.82,12.69,25.38,16.82,0.026267,0.236407
1,5.88,0.05,-1.00,8.99,17.98,14.00,0.037078,0.333704
2,3.02,0.03,-0.91,8.38,16.76,14.09,0.039777,0.357995
3,5.98,0.01,-0.66,27.49,54.98,14.34,0.012126,0.109131
4,5.99,0.05,1.97,8.87,17.74,16.97,0.037580,0.338219
5,3.06,0.05,1.32,6.28,12.56,16.32,0.053079,0.477707
6,4.12,0.01,-0.02,20.50,41.00,14.98,0.016260,0.146341
7,5.86,0.03,0.75,12.08,24.16,15.75,0.027594,0.248344
8,4.19,0.05,-0.17,7.59,15.18,14.83,0.043917,0.395257
9,5.98,0.01,1.96,24.62,49.24,16.96,0.013539,0.121852


In [ ]:
import os

def get_n_cells_from_owner(case_dir: str) -> int:
    """
    Reads the 'owner' file header in OpenFOAM polyMesh and extracts nCells.

    Parameters
    ----------
    case_dir : str
        Path to the root of the OpenFOAM case (should contain constant/polyMesh).

    Returns
    -------
    int
        Number of cells in the mesh.
    """
    owner_file = os.path.join(case_dir, "constant", "polyMesh", "owner")
    if not os.path.isfile(owner_file):
        raise FileNotFoundError(f"'owner' file not found at {owner_file}")

    with open(owner_file, "r") as f:
        for line in f:
            if "note" in line and "nCells" in line:
                match = re.search(r"nCells\s*:\s*(\d+)", line)
                if match:
                    return int(match.group(1))

    raise ValueError("Could not find nCells in the owner file header.")


case_path = "/nfs/home/geocean/alonsoap/projects/HyCFD/inputs/templates/openfoam"
print("Number of cells:", get_n_cells(case_path))

ValueError: invalid literal for int() with base 10: '/*--------------------------------*- C++ -*----------------------------------*\\'

In [ ]:
import os
import re

def get_n_cells(case_dir: str) -> int:
    """
    Return the number of cells in an OpenFOAM mesh.
    
    It reads the 'owner' file in constant/polyMesh and extracts nCells from the header.
    
    Parameters
    ----------
    case_dir : str
        Path to the OpenFOAM case directory.
    
    Returns
    -------
    int
        Number of cells in the mesh.
    """
    owner_file = os.path.join(case_dir, "constant", "polyMesh", "owner")
    
    if not os.path.isfile(owner_file):
        raise FileNotFoundError(f"'owner' file not found at {owner_file}")
    
    with open(owner_file, "r") as f:
        for line in f:
            # Look for the line containing nCells in the header
            if "note" in line and "nCells" in line:
                match = re.search(r"nCells\s*:\s*(\d+)", line)
                if match:
                    return int(match.group(1))
    
    raise ValueError("Could not find nCells in the owner file header.")

In [ ]:
case_path = "/nfs/home/geocean/alonsoap/projects/HyCFD/inputs/templates/openfoam"
print("Number of cells:", get_n_cells(case_path))

Number of cells: 114144
